In [ ]:
import uuid
from pathlib import Path
from urllib.parse import urlencode

import cv2
import magic
import requests
import yt_dlp

In [37]:
class VideoDownloader:
    def __init__(self, base_url: str, save_path: str = './'):
        self._base_url = base_url
        self._save_path = Path(save_path.rstrip('/'))
    
    def download_file(self, url: str, resolution: int | None = None) -> str:
        raise NotImplementedError


In [38]:
class YandexDiskDownloader(VideoDownloader):
    def __init__(self,
                 base_url: str = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?',
                 save_path: str = './'):
        super().__init__(base_url, save_path)

    def _get_download_link(self, url: str):
        get_url = self._base_url + urlencode(dict(public_key=url))
        response = requests.get(get_url)

        return response.json()['href']

    def _write_file_on_disk(self, response, file_path: str):
        with open(file_path, 'wb') as f:
            f.write(response.content)

    def download_file(self, url: str, resolution: int | None = None) -> str:
        download_link = self._get_download_link(url)
        download_response = requests.get(download_link)

        filename = self._get_file_name(download_response.content)
        file_path = self._save_path / filename

        self._write_file_on_disk(download_response, file_path)
        
        return str(file_path.absolute())

    def _get_file_name(self, content: bytes):
        file_extension = magic.from_buffer(content, mime=True).split('/')[-1]
        print(magic.from_buffer(content, mime=True))
        return f"{uuid.uuid4()}.{file_extension}"


In [ ]:
class RutubeDownloader(VideoDownloader):
    def __init__(self, save_path: str = './'):
        super().__init__('', save_path)
    
    def download_file(self, url: str, resolution: int | None = None) -> str:
        ydl_opts = {
            'outtmpl': str(self._save_path / '%(title)s.%(ext)s'),
            'merge_output_format': 'mp4',
        }
        if resolution:
            ydl_opts['format'] = f'bestvideo[height<={resolution}]+bestaudio/best[height<={resolution}]'

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False)
            filename = ydl.prepare_filename(info)
            file_path = Path(filename).with_suffix('.mp4')

            if file_path.exists():
                if input('Видео уже скачано. Скачать заново? ') == 'n':
                    return str(file_path.absolute())

            print('Video is downloading...')
            ydl.download([url])

        return str(file_path.absolute())


In [46]:
url = 'https://rutube.ru/video/d1f35432c70d3d9c941ca8663009fb5c/'
downloader = RutubeDownloader()
downloader.download_file(url)

Убежище — Русский трейлер (2026) (1920x1080) |████████████████████████████████████████| 42/42 [100%] in 56.2s (0.75/s) 


'c:\\Projects\\data-analysis\\video-detector\\Убежище — Русский трейлер (2026) (1920x1080).mp4'

In [ ]:
public_url = 'https://disk.yandex.ru/i/something'

disk = YandexDiskDownloader()
disk.download_file(public_url)

In [47]:
def split_to_frames(file_path: str, save_path: str, frame_interval: int):
    dir_name = file_path.split('/')[-1].split('.')[0]
    _save_path = Path(save_path) / dir_name
    
    if _save_path.exists():
        print(f'[WARNING] {str(_save_path.absolute())} already exists and will be recreated')
        _save_path.rmdir()

    _save_path.mkdir(parents=True)

    cap = cv2.VideoCapture(file_path)
    
    frame_counter = 0
    frame_index = 1
    while (r := cap.read())[0]:
        frame_counter += 1
        frame = r[1]
        
        if frame_counter % frame_interval == 0:
            frame_save_path = _save_path / f'frame_{frame_index}.jpg'
           
            success, encoded_img = cv2.imencode('.jpg', frame)
            
            if success:
                with open(frame_save_path, 'wb') as f:
                    f.write(encoded_img)

            frame_counter = 0
            frame_index += 1
    
    cap.release()

    print(f'[INFO] Всего сохранено {frame_index - 1} кадров')


In [49]:
split_to_frames('./Убежище — Русский трейлер (2026) (1920x1080).mp4', save_path='./frames', frame_interval=30)


[INFO] Всего сохранено 162 кадров
